In [ ]:
# default_exp traceability.unsupervised.approach.d2v

# Experimenting Neural Unsupervised Approaches for Software Information Retrieval [d2v]

> This module is dedicated to evaluate doc2vec. Consider to Copy the entire notebook for a new and separeted empirical evaluation. 
> Implementing mutual information analysis
> Author: @danaderp April 2020
> Author: @danielrc Nov 2020

In [1]:
#!pip install gensim
#!pip install seaborn
#!pip install sklearn
!pip install -e .

Obtaining file:///Users/loganfecko/Desktop/Summer_Research/traceXplainer/notebooks
ERROR: file:///Users/loganfecko/Desktop/Summer_Research/traceXplainer/notebooks does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


This copy is for Cisco purposes. It was adapted to process private github data from cisco. 

In [2]:
import numpy as np
import gensim
import pandas as pd
from itertools import product 
from random import sample 
import functools 
import os
from enum import Enum, unique, auto

In [3]:
#export
from datetime import datetime
#import seaborn as sns

In [4]:
#export
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

In [5]:
#export
from sklearn.metrics import precision_recall_curve
#from sklearn.metrics import plot_precision_recall_curve
from sklearn.metrics import auc
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
from pandas.plotting import lag_plot
import math as m
import random as r
import collections
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
#export
from gensim.similarities import SparseTermSimilarityMatrix, WordEmbeddingSimilarityIndex
from gensim import corpora

In [7]:
#https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.cosine.html
#export
from scipy.spatial import distance
from scipy.stats import pearsonr

In [8]:
#export
from sklearn.metrics import average_precision_score
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix

In [9]:
from pathlib import Path

In [10]:
import sys, os

# point Python at the parent of ds4se/
sys.path.insert(0, os.path.abspath(os.path.join('..', 'main')))


In [11]:
import ds4se as ds

In [12]:
from ds4se.mgmnt.prep.conv import *

# Experients Set-up

In [13]:
class VectorizationType(Enum):
    word2vec = auto()
    doc2vec = auto()
    vsm2vec = auto()

In [14]:
VectorizationType.doc2vec

<VectorizationType.doc2vec: 2>

In [15]:
class LinkType(Enum):
    req2tc = auto()
    req2src = auto()
    issue2src = auto()
    pr2src = auto()

In [16]:
class DistanceMetric(Enum):
    WMD = auto()
    COS = auto()
    SCM = auto()
    EUC = auto()
    MAN = auto()

In [17]:
class SimilarityMetric(Enum):
    WMD_sim = auto()
    COS_sim = auto()
    SCM_sim = auto()
    EUC_sim = auto()
    MAN_sim = auto()
    Pearson = auto()

In [18]:
class EntropyMetric(Enum):
    MSI_I = auto() #Minimum shared information Entropy
    MSI_X = auto()

In [19]:
class Preprocessing(Enum):
    conv = auto()
    bpe = auto()

In [20]:
LinkType.req2src

<LinkType.req2src: 2>

In [21]:
Preprocessing.bpe

<Preprocessing.bpe: 2>

In [22]:
path_data = '../dvc-data/systems/dronology' #dataset path

In [23]:

path_to_trained_model = '../dvc-data/systems/dronology/2vec_models/bpe32k/doc2vec/[doc2vec-py-java-PVDBOW-500-20E-32k-c-1751384704.291919].model'


In [24]:
#CISCO GitHub Parameters
def sacp_params():
    return {
        "vectorizationType": VectorizationType.word2vec,
        "linkType": LinkType.issue2src,
        "system": 'sacp-python-common',
        "path_to_trained_model": path_data + 'models/wv/conv/[word2vec-Py-Java-Wiki-SK-500-20E[0]-1592979270.711115].model',
        "source_type": SoftwareArtifacts.PR,
        "target_type": SoftwareArtifacts.PY,
        "path_mappings": '/tf/data/cisco/sacp_data/sacp-pr-mappings.csv',
        "system_path_config": {
            "system_path": '/tf/data/cisco/sacp_data/[sacp-python-common-all-corpus-1596383717.992744].csv', #MUST have bpe8k <----
            "sep": '~',
            "names": ['ids','conv'],
            "prep": Preprocessing.conv
        },
        "saving_path":  path_data/'se-benchmarking/traceability/cisco/sacp',
        "names": ['Source','Target','Linked?']
    }

In [25]:
def sacp_params_bpe():
    return {
        "vectorizationType": VectorizationType.word2vec,
        "linkType": LinkType.issue2src,
        "system": 'sacp-python-common',
        "path_to_trained_model": path_to_trained_model,
        "source_type": SoftwareArtifacts.PR,
        "target_type": SoftwareArtifacts.PY,
        "path_mappings": '/tf/data/cisco/sacp_data/sacp-pr-mappings.csv',
        "system_path_config": {
            "system_path": '/tf/data/cisco/sacp_data/[sacp-python-common-all-corpus-1596383717.992744].csv',
            "sep": '~',
            "names": ['ids','bpe8k'],
            "prep": Preprocessing.bpe
        },
        "saving_path": path_data + 'se-benchmarking/traceability/cisco/sacp',
        "names": ['Source','Target','Linked?'],
        "model_prefix":path_data + 'models/bpe/sentencepiece/wiki_py_java_bpe_8k' #For BPE Analysis
    }

In [26]:
def dronology_params():
    return {
        "vectorizationType": VectorizationType.doc2vec,
        "linkType": LinkType.req2src,
        "system": 'dronology',
        "path_to_trained_model": path_to_trained_model,
        "source_type": 'req',
        "target_type": 'code',
        "path_mappings": '../dvc-data/systems/dronology/mappings/conventional.csv',
        "system_path_config":{
            "system_path": '../dvc-data/systems/dronology/corpus/dronology_conv_with_bpe_fixed.csv',
            "sep": '~',
            #"index_col": False,
            #"header": 0,
            "names": ['ids','bpe128k'],
            "prep": Preprocessing.bpe,
            "bpe_model_path": '../dvc-data/systems/dronology/bpe_models/java_py_bpe128k.model'
        },
        "saving_path": '../dvc-data/systems/dronology/experiment/4.1.5/results/',
        "names": ['Source','Target','Linked?']
    }

In [27]:

#parameters = sacp_params_bpe()
#parameters = sacp_params()
parameters = dronology_params()
parameters

{'vectorizationType': <VectorizationType.doc2vec: 2>,
 'linkType': <LinkType.req2src: 2>,
 'system': 'dronology',
 'path_to_trained_model': '../dvc-data/systems/dronology/2vec_models/bpe32k/doc2vec/[doc2vec-py-java-PVDBOW-500-20E-32k-c-1751384704.291919].model',
 'source_type': 'req',
 'target_type': 'code',
 'path_mappings': '../dvc-data/systems/dronology/mappings/conventional.csv',
 'system_path_config': {'system_path': '../dvc-data/systems/dronology/corpus/dronology_conv_with_bpe_fixed.csv',
  'sep': '~',
  'names': ['ids', 'bpe128k'],
  'prep': <Preprocessing.bpe: 2>,
  'bpe_model_path': '../dvc-data/systems/dronology/bpe_models/java_py_bpe128k.model'},
 'saving_path': '../dvc-data/systems/dronology/experiment/4.1.5/results/',
 'names': ['Source', 'Target', 'Linked?']}

### Testing experiments set-up

In [28]:
#tst
parameters['system_path_config']['system_path']

'../dvc-data/systems/dronology/corpus/dronology_conv_with_bpe_fixed.csv'

In [29]:
#tst
parameters['system_path_config']['names'][1]

'bpe128k'

In [30]:
parameters['system_path_config']['sep'] #tst

'~'

In [31]:
#tst
df_all_system = pd.read_csv(
            parameters['system_path_config']['system_path'], 
            #names = params['system_path_config']['names'], #include the names into the files!!!
            header = 0, 
            index_col = 0, 
            sep = "~"
    )

In [32]:
df_all_system.head(1)

,filenames,text,type,conv,bpe8k,bpe32k,bpe128k
ids,,,,,,,
/Users/loganfecko/Desktop/Summer_Research/traceXplainer/dvc-data/systems/dronology/corpus/req/RE-616.txt,RE-616.txt,[SUMMARY]\nLogging of GCS related events\n[DES...,req,summari log gcs relat event descript activ log...,▁sum m ari ▁log ▁g cs ▁rel at ▁event ▁descri p...,▁sum m ari ▁log ▁g cs ▁rel at ▁event ▁descri p...,▁summ ari ▁log ▁gcs ▁relat ▁event ▁descri pt ▁...


In [33]:
#tst
tag = parameters['system_path_config']['names'][1]
[doc.split() for doc in df_all_system[df_all_system[tag].notnull()][tag].values]

[['▁summ',
  'ari',
  '▁log',
  '▁gcs',
  '▁relat',
  '▁event',
  '▁descri',
  'pt',
  '▁activ',
  '▁logger',
  '▁shall',
  '▁log',
  '▁command',
  '▁sent',
  '▁mess',
  'ag',
  '▁rece',
  'iv',
  '▁cgs'],
 ['▁summ',
  'ari',
  '▁support',
  '▁gcs',
  '▁connect',
  '▁u',
  'av',
  '▁descri',
  'pt',
  '▁veh',
  'ic',
  'l',
  '▁core',
  '▁shall',
  '▁support',
  '▁u',
  'av',
  '▁connect',
  '▁via',
  '▁gcs'],
 ['▁summ',
  'ari',
  '▁manual',
  '▁flight',
  '▁plan',
  '▁assign',
  '▁descri',
  'pt',
  '▁rout',
  '▁contain',
  '▁spec',
  'if',
  '▁u',
  'av',
  '▁sing',
  'l',
  '▁u',
  'av',
  'flight',
  '▁plan',
  '▁sched',
  'ul',
  '▁shall',
  '▁assign',
  '▁rout',
  '▁given',
  '▁u',
  'av'],
 ['▁summ',
  'ari',
  '▁visit',
  '▁waypoint',
  '▁flight',
  '▁plan',
  '▁descri',
  'pt',
  '▁flight',
  '▁plan',
  '▁execut',
  '▁veh',
  'ic',
  'l',
  '▁core',
  '▁shall',
  '▁send',
  '▁next',
  '▁waypoint',
  '▁u',
  'av'],
 ['▁summ',
  'ari',
  '▁establish',
  '▁maximum',
  '▁number',

In [34]:
len(df_all_system[tag].values) #tst

242

In [35]:
#tst
len(df_all_system[df_all_system[tag].notnull()]) #some files are _init_ thefore are empty

242

In [36]:
#tst
df_all_system[df_all_system[tag].notnull()][tag].values

array(['▁summ ari ▁log ▁gcs ▁relat ▁event ▁descri pt ▁activ ▁logger ▁shall ▁log ▁command ▁sent ▁mess ag ▁rece iv ▁cgs',
       '▁summ ari ▁support ▁gcs ▁connect ▁u av ▁descri pt ▁veh ic l ▁core ▁shall ▁support ▁u av ▁connect ▁via ▁gcs',
       '▁summ ari ▁manual ▁flight ▁plan ▁assign ▁descri pt ▁rout ▁contain ▁spec if ▁u av ▁sing l ▁u av flight ▁plan ▁sched ul ▁shall ▁assign ▁rout ▁given ▁u av',
       '▁summ ari ▁visit ▁waypoint ▁flight ▁plan ▁descri pt ▁flight ▁plan ▁execut ▁veh ic l ▁core ▁shall ▁send ▁next ▁waypoint ▁u av',
       '▁summ ari ▁establish ▁maximum ▁number ▁activ ▁u av ▁descri pt ▁maximum ▁number ▁allow ▁activ ▁u av ▁reach ▁u av activ ▁man ag ▁shall ▁reject ▁activ ▁request',
       '▁summ ari ▁support ▁intern ▁simul ▁u av ▁descri pt ▁veh ic l ▁core ▁shall ▁support ▁virtual ▁u av',
       '▁summ ari ▁middle war ▁list ▁save ▁flight ▁rout ▁descri pt ▁u im iddle war ▁shall ▁provid ▁list ▁exist ▁rout',
       '▁summ ari ▁mission ▁plan ▁descri pt ▁mission ▁planner ▁shall ▁ex

In [37]:
df_all_system.reset_index(inplace=True)

In [38]:
#tst
df_all_system.loc[df_all_system['type'] == parameters['source_type']][parameters['system_path_config']['names']]

,ids,bpe128k
0,/Users/loganfecko/Desktop/Summer_Research/trac...,▁summ ari ▁log ▁gcs ▁relat ▁event ▁descri pt ▁...
1,/Users/loganfecko/Desktop/Summer_Research/trac...,▁summ ari ▁support ▁gcs ▁connect ▁u av ▁descri...
2,/Users/loganfecko/Desktop/Summer_Research/trac...,▁summ ari ▁manual ▁flight ▁plan ▁assign ▁descr...
3,/Users/loganfecko/Desktop/Summer_Research/trac...,▁summ ari ▁visit ▁waypoint ▁flight ▁plan ▁desc...
4,/Users/loganfecko/Desktop/Summer_Research/trac...,▁summ ari ▁establish ▁maximum ▁number ▁activ ▁...
5,/Users/loganfecko/Desktop/Summer_Research/trac...,▁summ ari ▁support ▁intern ▁simul ▁u av ▁descr...
6,/Users/loganfecko/Desktop/Summer_Research/trac...,▁summ ari ▁middle war ▁list ▁save ▁flight ▁rou...
7,/Users/loganfecko/Desktop/Summer_Research/trac...,▁summ ari ▁mission ▁plan ▁descri pt ▁mission ▁...
8,/Users/loganfecko/Desktop/Summer_Research/trac...,▁summ ari ▁log ▁flight ▁plan ▁relat ▁event ▁de...
9,/Users/loganfecko/Desktop/Summer_Research/trac...,▁summ ari ▁accept ▁mission ▁plan ▁descri pt ▁u...


In [39]:
df_all_system.loc[df_all_system['type'] == parameters['target_type']][parameters['system_path_config']['names']]

,ids,bpe128k
58,/Users/loganfecko/Desktop/Summer_Research/trac...,▁packag ▁edu ▁d ron olog ▁servic ▁ext ens ▁mis...
59,/Users/loganfecko/Desktop/Summer_Research/trac...,▁packag ▁edu ▁d ron olog ▁servic ▁ext ens ▁mis...
60,/Users/loganfecko/Desktop/Summer_Research/trac...,▁packag ▁edu ▁d ron olog ▁servic ▁core ▁util ▁...
61,/Users/loganfecko/Desktop/Summer_Research/trac...,▁packag ▁edu ▁d ron olog ▁core ▁veh ic l ▁inte...
62,/Users/loganfecko/Desktop/Summer_Research/trac...,▁packag ▁edu ▁d ron olog ▁servic ▁ext ens ▁mis...
...,...,...
237,/Users/loganfecko/Desktop/Summer_Research/trac...,▁packag ▁edu ▁d ron olog ▁core ▁veh ic l ▁comm...
238,/Users/loganfecko/Desktop/Summer_Research/trac...,▁packag ▁edu ▁d ron olog ▁va adin ▁active flig...
239,/Users/loganfecko/Desktop/Summer_Research/trac...,▁packag ▁edu ▁d ron olog ▁va adin ▁active flig...
240,/Users/loganfecko/Desktop/Summer_Research/trac...,▁packag ▁edu ▁d ron olog ▁servic ▁ext ens ▁mis...


### Defining BasicSequenceVectorization

In [40]:
#tst
print(list(VectorizationType), list(DistanceMetric), list(SimilarityMetric), list(LinkType))

[<VectorizationType.word2vec: 1>, <VectorizationType.doc2vec: 2>, <VectorizationType.vsm2vec: 3>] [<DistanceMetric.WMD: 1>, <DistanceMetric.COS: 2>, <DistanceMetric.SCM: 3>, <DistanceMetric.EUC: 4>, <DistanceMetric.MAN: 5>] [<SimilarityMetric.WMD_sim: 1>, <SimilarityMetric.COS_sim: 2>, <SimilarityMetric.SCM_sim: 3>, <SimilarityMetric.EUC_sim: 4>, <SimilarityMetric.MAN_sim: 5>, <SimilarityMetric.Pearson: 6>] [<LinkType.req2tc: 1>, <LinkType.req2src: 2>, <LinkType.issue2src: 3>, <LinkType.pr2src: 4>]


In [41]:
#export
class BasicSequenceVectorization():
    '''Implementation of the class sequence-vanilla-vectorization other classes can inheritance this one'''
    def __init__(self, params):
                
        self.params = params
        self.df_nonground_link = None
        self.df_ground_link = None
        self.prep = ConventionalPreprocessing(params, bpe = False)
        
        self.df_all_system = pd.read_csv(
            params['system_path_config']['system_path'], 
            #names = params['system_path_config']['names'], #include the names into the files!!!
            header = 0, 
            #index_col = 0, 
            sep = params['system_path_config']['sep'] 
        )
        
        #self.df_source = pd.read_csv(params['source_path'], names=['ids', 'text'], header=None, sep=' ')
        #self.df_target = pd.read_csv(params['target_path'], names=['ids', 'text'], header=None, sep=' ')
        self.df_source = self.df_all_system.loc[self.df_all_system['type'] == params['source_type']][params['system_path_config']['names']]
        self.df_target = self.df_all_system.loc[self.df_all_system['type'] == params['target_type']][params['system_path_config']['names']]
        
        #NA verification
        tag = parameters['system_path_config']['names'][1]
        self.df_source[tag] = self.df_source[tag].fillna("")
        self.df_target[tag] = self.df_target[tag].fillna("")
        
        if params['system_path_config']['prep'] == Preprocessing.conv: #if conventional preprocessing
            self.documents = [doc.split() for doc in self.df_all_system[self.df_all_system[tag].notnull()][tag].values] #Preparing Corpus
            self.dictionary = corpora.Dictionary( self.documents ) #Preparing Dictionary
            logging.info("conventional preprocessing documents and dictionary")
            self.vocab = self.dictionary.token2id
        elif params['system_path_config']['prep'] == Preprocessing.bpe:
            from sentencepiece import SentencePieceProcessor
            sp = SentencePieceProcessor()
            sp.load(params['system_path_config']['bpe_model_path'])
            self.prep.sp_bpe = sp
            
            
            self.documents = [doc.split() for doc in self.df_all_system[tag].values]            
            self.dictionary = corpora.Dictionary( self.documents ) #Preparing Dictionary
            logging.info("bpe preprocessing documents and dictionary")
                
            ####INFO science params
            abstracted_vocab = [ set(doc) for doc in self.df_all_system[ 'bpe8k' ].values] #creation of sets
            abstracted_vocab = functools.reduce( lambda a,b : a.union(b), abstracted_vocab ) #union of sets
            self.vocab = {self.prep.sp_bpe.id_to_piece(id): 0 for id in range(self.prep.sp_bpe.get_piece_size())}
            dict_abs_vocab = { elem : 0 for elem in abstracted_vocab - set(self.vocab.keys()) } #Ignored vocab by BPE
            self.vocab.update(dict_abs_vocab) #Updating
        
        
        #This can be extended for future metrics <---------------------
        #TODO include mutual and join information
        self.dict_labels = {
            DistanceMetric.COS:[DistanceMetric.COS, SimilarityMetric.COS_sim],
            SimilarityMetric.Pearson:[SimilarityMetric.Pearson],
            DistanceMetric.EUC:[DistanceMetric.EUC, SimilarityMetric.EUC_sim],
            DistanceMetric.WMD:[DistanceMetric.WMD, SimilarityMetric.WMD_sim],
            DistanceMetric.SCM:[DistanceMetric.SCM, SimilarityMetric.SCM_sim],
            DistanceMetric.MAN:[DistanceMetric.MAN, SimilarityMetric.MAN_sim],
            EntropyMetric.MSI_I:[EntropyMetric.MSI_I, EntropyMetric.MSI_X],
            #EntropyMetric.MI:[EntropyMetric.JI, EntropyMetric.MI]
        }

        
    def ground_truth_processing(self, path_to_ground_truth = '', from_mappings = False):
        'Optional class when corpus has ground truth. This function create tuples of links'
        
        if from_mappings:
            df_mapping = pd.read_csv(self.params['path_mappings'], header = 0, sep = ',')
            ground_links = list(zip(df_mapping['id_pr'].astype(str), df_mapping['doc_id']))
        else:
            ground_truth = open(path_to_ground_truth,'r')
            #Organizing The Ground Truth under the given format
            ground_links = [ [(line.strip().split()[0], elem) for elem in line.strip().split()[1:]] for line in ground_truth]
            ground_links = functools.reduce(lambda a,b : a+b,ground_links) #reducing into one list
            assert len(ground_links) ==  len(set(ground_links)) #To Verify Redundancies in the file
        return ground_links
    
    def samplingLinks(self, sampling = False, samples = 10, basename = False):
        
        if basename:
            source = [os.path.basename(elem) for elem in self.df_source['ids'].values ] 
            target = [os.path.basename(elem) for elem in self.df_target['ids'].values ]
        else:
            source = self.df_source['ids'].values
            target = self.df_target['ids'].values

        if sampling:
            links = sample( list( product( source , target ) ), samples)
        else:
            links = list( product( source , target ))

        return links
    
    def cos_scipy(self, vector_v, vector_w):
        cos =  distance.cosine( vector_v, vector_w )
        return [cos, 1.-cos]
    
    def euclidean_scipy(self, vector_v, vector_w):
        dst = distance.euclidean(vector_v,vector_w)
        return [dst, 1./(1.+dst)] #Computing the inverse for similarity
    
    def manhattan_scipy(self, vector_v, vector_w):
        dst = distance.cityblock(vector_v,vector_w)
        n = len(vector_v)
        return [dst, 1./(1.+dst)] #Computing the inverse for similarity
    
    def pearson_abs_scipy(self, vector_v, vector_w):
        '''We are not sure that pearson correlation works well on doc2vec inference vectors'''
        #vector_v =  np.asarray(vector_v, dtype=np.float32)
        #vector_w =  np.asarray(vector_w, dtype=np.float32)
        logging.info("pearson_abs_scipy" + str(vector_v) + "__" + str(vector_w))
        corr, _ = pearsonr(vector_v, vector_w)
        return [abs(corr)] #Absolute value of the correlation
    

    def computeDistanceMetric(self, links, metric_list):
        '''Metric List Iteration''' 
        
        metric_labels = [ self.dict_labels[metric] for metric in metric_list] #tracking of the labels
        distSim = [[link[0], link[1], self.distance( metric_list, link )] for link in links] #Return the link with metrics
        distSim = [[elem[0], elem[1]] + elem[2] for elem in distSim] #Return the link with metrics
        
        return distSim, functools.reduce(lambda a,b : a+b, metric_labels)
    
    def ComputeDistanceArtifacts(self, metric_list, sampling = False , samples = 10, basename = False):
        '''Activates Distance and Similarity Computations
        @metric_list if [] then Computes All metrics
        @sampling is False by the default
        @samples is the number of samples (or links) to be generated'''
        links_ = self.samplingLinks( sampling, samples, basename )
        
        docs, metric_labels = self.computeDistanceMetric( metric_list=metric_list, links=links_) #checkpoints
        self.df_nonground_link = pd.DataFrame(docs, columns =[self.params['names'][0], self.params['names'][1]]+ metric_labels) #Transforming into a Pandas
        logging.info("Non-groundtruth links computed")
        pass 
    
    
    def SaveLinks(self, grtruth=False, sep=' ', mode='a'):
        timestamp = datetime.timestamp(datetime.now())
        path_to_link = self.params['saving_path'] + '['+ self.params['system'] + '-' + str(self.params['vectorizationType']) + '-' + str(self.params['linkType']) + '-' + str(grtruth) + '-{}].csv'.format(timestamp)
        
        if grtruth:
            self.df_ground_link.to_csv(path_to_link, header=True, index=True, sep=sep, mode=mode)
        else:
            self.df_nonground_link.to_csv(path_to_link, header=True, index=True, sep=sep, mode=mode)
        
        logging.info('Saving in...' + path_to_link)
        pass
    
    def findDistInDF(self, g_tuple, from_mappings=False, semeru_format=False):
        '''Return the index values of the matched mappings
        .eq is used for Source since it must match the exact code to avoid number substrings
        for the target, the substring might works fine'''

        if from_mappings:
            dist = self.df_ground_link.loc[(self.df_ground_link["Source"].eq(g_tuple[0]) ) & 
                 (self.df_ground_link["Target"].str.contains(g_tuple[1], regex=False))]
            logging.info('findDistInDF: from_mappings')
        elif semeru_format:
            dist = self.df_ground_link.loc[(self.df_ground_link["Source"].str.contains(g_tuple[0], regex=False) ) & 
                 (self.df_ground_link["Target"].str.contains(g_tuple[1], regex=False))]
            logging.info('findDistInDF: semeru_format')
        else:
            dist = self.df_ground_link[self.df_ground_link[self.params['names'][0]].str.contains( g_tuple[0][:g_tuple[0].find('.')] + '-' ) 
                     & self.df_ground_link[self.params['names'][1]].str.contains(g_tuple[1][:g_tuple[1].find('.')]) ]
            logging.info('findDistInDF: default')
        return dist.index.values
    
        
    def MatchWithGroundTruth(self, path_to_ground_truth='', from_mappings=False, semeru_format=False ):
        self.df_ground_link = self.df_nonground_link.copy()
        self.df_ground_link[self.params['names'][2]] = 0
        
        matchGT = [ self.findDistInDF( g , from_mappings=from_mappings, semeru_format=semeru_format ) for g in self.ground_truth_processing(path_to_ground_truth,from_mappings)]
        matchGT = functools.reduce(lambda a,b : np.concatenate([a,b]), matchGT) #Concatenate indexes
        new_column = pd.Series(np.full([len(matchGT)], 1 ), name=self.params['names'][2], index = matchGT)
        
        self.df_ground_link.update(new_column)
        logging.info("Groundtruth links computed")
        pass

### Testing BasicSequenceVectorization

In [42]:
general2vec =  BasicSequenceVectorization(params = parameters)

2025-07-06 11:27:07,253 : INFO : adding document #0 to Dictionary<0 unique tokens: []>
2025-07-06 11:27:07,271 : INFO : built Dictionary<1652 unique tokens: ['ag', 'ari', 'iv', 'pt', '▁activ']...> from 242 documents (total 78439 corpus positions)
2025-07-06 11:27:07,271 : INFO : Dictionary lifecycle event {'msg': "built Dictionary<1652 unique tokens: ['ag', 'ari', 'iv', 'pt', '▁activ']...> from 242 documents (total 78439 corpus positions)", 'datetime': '2025-07-06T11:27:07.271744', 'gensim': '4.3.3', 'python': '3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 18:02:02) \n[Clang 18.1.8 ]', 'platform': 'macOS-15.5-arm64-arm-64bit', 'event': 'created'}
2025-07-06 11:27:07,272 : INFO : bpe preprocessing documents and dictionary


In [43]:
general2vec.vocab

{'<pad>': 0,
 '<unk>': 0,
 '<s>': 0,
 '</s>': 0,
 '▁t': 0,
 'in': 0,
 'er': 0,
 '▁a': 0,
 '▁.': 0,
 'on': 0,
 '▁(': 0,
 '▁)': 0,
 '▁,': 0,
 'he': 0,
 're': 0,
 '▁s': 0,
 'at': 0,
 'en': 0,
 'or': 0,
 '▁the': 0,
 '▁c': 0,
 '▁=': 0,
 'es': 0,
 '▁f': 0,
 'ed': 0,
 'al': 0,
 'is': 0,
 '▁p': 0,
 'an': 0,
 'it': 0,
 'ar': 0,
 '▁w': 0,
 '▁o': 0,
 '▁in': 0,
 'el': 0,
 'et': 0,
 'ing': 0,
 'ion': 0,
 '▁b': 0,
 '▁d': 0,
 '▁re': 0,
 'as': 0,
 'ro': 0,
 '▁n': 0,
 'nd': 0,
 'ic': 0,
 'le': 0,
 '▁m': 0,
 '▁;': 0,
 'ur': 0,
 '▁of': 0,
 'am': 0,
 '▁to': 0,
 '▁"': 0,
 '▁and': 0,
 'ent': 0,
 '▁:': 0,
 '▁l': 0,
 '▁g': 0,
 '▁h': 0,
 '▁i': 0,
 '▁S': 0,
 'ct': 0,
 "▁'": 0,
 'ut': 0,
 'st': 0,
 'ce': 0,
 'il': 0,
 '▁T': 0,
 '▁th': 0,
 'ul': 0,
 '▁e': 0,
 'id': 0,
 'ad': 0,
 '▁{': 0,
 '▁1': 0,
 'om': 0,
 'se': 0,
 '▁}': 0,
 'tr': 0,
 '▁C': 0,
 '▁A': 0,
 'ot': 0,
 'ol': 0,
 '▁for': 0,
 '▁if': 0,
 'un': 0,
 '▁[': 0,
 'ig': 0,
 '▁]': 0,
 'od': 0,
 'elf': 0,
 '▁v': 0,
 'im': 0,
 '▁self': 0,
 'ch': 0,
 'pt': 0,
 '

In [44]:
general2vec.documents

[['▁summ',
  'ari',
  '▁log',
  '▁gcs',
  '▁relat',
  '▁event',
  '▁descri',
  'pt',
  '▁activ',
  '▁logger',
  '▁shall',
  '▁log',
  '▁command',
  '▁sent',
  '▁mess',
  'ag',
  '▁rece',
  'iv',
  '▁cgs'],
 ['▁summ',
  'ari',
  '▁support',
  '▁gcs',
  '▁connect',
  '▁u',
  'av',
  '▁descri',
  'pt',
  '▁veh',
  'ic',
  'l',
  '▁core',
  '▁shall',
  '▁support',
  '▁u',
  'av',
  '▁connect',
  '▁via',
  '▁gcs'],
 ['▁summ',
  'ari',
  '▁manual',
  '▁flight',
  '▁plan',
  '▁assign',
  '▁descri',
  'pt',
  '▁rout',
  '▁contain',
  '▁spec',
  'if',
  '▁u',
  'av',
  '▁sing',
  'l',
  '▁u',
  'av',
  'flight',
  '▁plan',
  '▁sched',
  'ul',
  '▁shall',
  '▁assign',
  '▁rout',
  '▁given',
  '▁u',
  'av'],
 ['▁summ',
  'ari',
  '▁visit',
  '▁waypoint',
  '▁flight',
  '▁plan',
  '▁descri',
  'pt',
  '▁flight',
  '▁plan',
  '▁execut',
  '▁veh',
  'ic',
  'l',
  '▁core',
  '▁shall',
  '▁send',
  '▁next',
  '▁waypoint',
  '▁u',
  'av'],
 ['▁summ',
  'ari',
  '▁establish',
  '▁maximum',
  '▁number',

In [45]:
general2vec.dictionary

In [46]:
general2vec.df_all_system.head(1)

,ids,filenames,text,type,conv,bpe8k,bpe32k,bpe128k
0,/Users/loganfecko/Desktop/Summer_Research/trac...,RE-616.txt,[SUMMARY]\nLogging of GCS related events\n[DES...,req,summari log gcs relat event descript activ log...,▁sum m ari ▁log ▁g cs ▁rel at ▁event ▁descri p...,▁sum m ari ▁log ▁g cs ▁rel at ▁event ▁descri p...,▁summ ari ▁log ▁gcs ▁relat ▁event ▁descri pt ▁...


In [47]:
general2vec.df_all_system.shape #data final tensor

(242, 8)

In [48]:
#tst for libest
path_to_ground_truth = '../dvc-data/systems/dronology/mappings/ground_truth.txt'
general2vec.ground_truth_processing(path_to_ground_truth)

[('RE-100.txt', 'PlanPool.java'),
 ('RE-101.txt', 'FlightZoneManager2.java'),
 ('RE-101.txt', 'PlanPoolManager.java'),
 ('RE-103.txt', 'FlightZoneManager2.java'),
 ('RE-114.txt', 'AFMainLayout.java'),
 ('RE-114.txt', 'AFMapComponent.java'),
 ('RE-25.txt', 'AbstractDrone.java'),
 ('RE-25.txt', 'PhysicalDrone.java'),
 ('RE-25.txt', 'PhysicalDroneFleetFactory.java'),
 ('RE-25.txt', 'VirtualDrone.java'),
 ('RE-25.txt', 'VirtualDroneFleetFactory.java'),
 ('RE-28.txt', 'PlanPool.java'),
 ('RE-28.txt', 'PlanPoolManager.java'),
 ('RE-36.txt', 'DroneFleetManager.java'),
 ('RE-36.txt', 'PhysicalDroneFleetFactory.java'),
 ('RE-36.txt', 'VirtualDroneFleetFactory.java'),
 ('RE-424.txt', 'FlightZoneManager2.java'),
 ('RE-501.txt', 'DroneSimulatorServiceInstance.java'),
 ('RE-501.txt', 'DronologyPersistenceUtil.java'),
 ('RE-501.txt', 'FlightRoute.java'),
 ('RE-501.txt', 'FlightRouteInfo.java'),
 ('RE-501.txt', 'FlightRoutePersistenceProvider.java'),
 ('RE-501.txt', 'FlightRouteplanningService.java')

In [49]:
#tst for sacp
general2vec.ground_truth_processing(parameters['path_mappings'], from_mappings = True)

[('RE-100', 'PlanPool'),
 ('RE-101', 'FlightZoneManager2'),
 ('RE-101', 'PlanPoolManager'),
 ('RE-103', 'FlightZoneManager2'),
 ('RE-114', 'AFMainLayout'),
 ('RE-114', 'AFMapComponent'),
 ('RE-25', 'AbstractDrone'),
 ('RE-25', 'PhysicalDrone'),
 ('RE-25', 'PhysicalDroneFleetFactory'),
 ('RE-25', 'VirtualDrone'),
 ('RE-25', 'VirtualDroneFleetFactory'),
 ('RE-28', 'PlanPool'),
 ('RE-28', 'PlanPoolManager'),
 ('RE-36', 'DroneFleetManager'),
 ('RE-36', 'PhysicalDroneFleetFactory'),
 ('RE-36', 'VirtualDroneFleetFactory'),
 ('RE-424', 'FlightZoneManager2'),
 ('RE-501', 'DroneSimulatorServiceInstance'),
 ('RE-501', 'DronologyPersistenceUtil'),
 ('RE-501', 'FlightRoute'),
 ('RE-501', 'FlightRouteInfo'),
 ('RE-501', 'FlightRoutePersistenceProvider'),
 ('RE-501', 'FlightRouteplanningService'),
 ('RE-501', 'FlightRouteplanningServiceInstance'),
 ('RE-501', 'FlightRouteplanningServiceRemoteFacade'),
 ('RE-501', 'IFlightRoute'),
 ('RE-501', 'IFlightRouteplanningServiceInstance'),
 ('RE-501', 'Waypo

In [50]:
import math
import dit

# Artifacts Similarity with Doc2Vec

Try to reproduce the same empirical evaluation like here: [link](https://arxiv.org/pdf/1507.07998.pdf). Pay attention to:
- Accuracy vs. Dimensionality (we can replace accuracy for false positive rate or true positive rate)
- Visualize paragraph vectors using t-sne
- Computing Cosine Distance and Similarity. More about similarity [link](https://www.kdnuggets.com/2017/08/comparing-distance-measurements-python-scipy.html)

In [51]:
#path_to_trained_model": 'test_data/models/pv/conv/[doc2vec-Py-Java-PVDBOW-500-20E-1592609630.689167].model',
#"path_to_trained_model": 'test_data/models/pv/conv/[doc2vec-Py-Java-Wiki-PVDBOW-500-20E[15]-1592941134.367976].model',
path_to_trained_model = '../dvc-data/systems/dronology/2vec_models/bpe128k/doc2vec/[doc2vec-py-java-PVDBOW-500-20E-128k-c-1751384704.291919].model'

In [52]:

print("Columns:", df_all_system.columns.tolist())
print("Unique type values:", df_all_system['type'].unique())

Columns: ['ids', 'filenames', 'text', 'type', 'conv', 'bpe8k', 'bpe32k', 'bpe128k']
Unique type values: ['req' 'code']


In [53]:
def doc2vec_params():
    return {
        "vectorizationType": VectorizationType.doc2vec,
        "linkType": LinkType.req2src,
        "system": 'dronology',
        "source_type": 'req',
        "target_type": 'code',
        "path_to_trained_model": path_to_trained_model,
        #"source_path": '/tf/main/benchmarking/traceability/testbeds/nltk/[libest-pre-req].csv',
        #"target_path": '/tf/main/benchmarking/traceability/testbeds/nltk/[libest-pre-tc].csv',
        "system_path_config": {
            "system_path": "../dvc-data/systems/dronology/corpus/dronology_conv_with_bpe_fixed.csv",
            "names": ["ids","filenames","text","type", "bpe128k"],
            "sep": "~",
            "prep": Preprocessing.conv
        },
        "saving_path": '../dvc-data/systems/dronology/experiment4.1.5/results/',
        "names": ['Source','Target','Linked?']
    }

In [54]:
doc2vec_params = doc2vec_params()
doc2vec_params

{'vectorizationType': <VectorizationType.doc2vec: 2>,
 'linkType': <LinkType.req2src: 2>,
 'system': 'dronology',
 'source_type': 'req',
 'target_type': 'code',
 'path_to_trained_model': '../dvc-data/systems/dronology/2vec_models/bpe128k/doc2vec/[doc2vec-py-java-PVDBOW-500-20E-128k-c-1751384704.291919].model',
 'system_path_config': {'system_path': '../dvc-data/systems/dronology/corpus/dronology_conv_with_bpe_fixed.csv',
  'names': ['ids', 'filenames', 'text', 'type', 'bpe128k'],
  'sep': '~',
  'prep': <Preprocessing.conv: 1>},
 'saving_path': '../dvc-data/systems/dronology/experiment4.1.5/results/',
 'names': ['Source', 'Target', 'Linked?']}

In [55]:
#Export
class Doc2VecSeqVect(BasicSequenceVectorization):
    
    def __init__(self, params):
        super().__init__(params)
        self.new_model = gensim.models.Doc2Vec.load( params['path_to_trained_model'] )
        self.new_model.init_sims(replace=True)  # Normalizes the vectors in the word2vec class.
        self.df_inferred_src = None
        self.df_inferred_trg = None
        
        self.dict_distance_dispatcher = {
            DistanceMetric.COS: self.cos_scipy,
            SimilarityMetric.Pearson: self.pearson_abs_scipy,
            DistanceMetric.EUC: self.euclidean_scipy,
            DistanceMetric.MAN: self.manhattan_scipy
        }
    
    def distance(self, metric_list, link):
        '''Iterate on the metrics'''
        ν_inferredSource = (
            self.df_inferred_src
                .loc[self.df_inferred_src['ids'].str.contains(link[0]), 'inf-doc2vec']
                .iloc[0]
        )
        w_inferredTarget = (
    self.df_inferred_trg
        .loc[self.df_inferred_trg['ids'].str.contains(link[1]), 'inf-doc2vec']
        .iloc[0]
)
        
        dist = [ self.dict_distance_dispatcher[metric](ν_inferredSource,w_inferredTarget) for metric in metric_list]
        logging.info("Computed distances or similarities "+ str(link) + str(dist))    
        return functools.reduce(lambda a,b : a+b, dist) #Always return a list
    
    def computeDistanceMetric(self, links, metric_list):
        '''It is computed the cosine similarity'''
        
        metric_labels = [ self.dict_labels[metric] for metric in metric_list] #tracking of the labels
        distSim = [[link[0], link[1], self.distance( metric_list, link )] for link in links] #Return the link with metrics
        distSim = [[elem[0], elem[1]] + elem[2] for elem in distSim] #Return the link with metrics
        
        return distSim, functools.reduce(lambda a,b : a+b, metric_labels)

    
    def InferDoc2Vec(self, steps=200):
        '''Activate Inference on Target and Source Corpus'''
        self.df_inferred_src = self.df_source.copy()
        self.df_inferred_trg = self.df_target.copy()
        
        self.df_inferred_src['inf-doc2vec'] =  [self.new_model.infer_vector(artifact.split(),epochs=steps) for artifact in self.df_inferred_src['text'].values]
        self.df_inferred_trg['inf-doc2vec'] =  [self.new_model.infer_vector(artifact.split(),epochs=steps) for artifact in self.df_inferred_trg['text'].values]
        
        logging.info("Infer Doc2Vec on Source and Target Complete")
    

### Testing Doc2Vec SequenceVectorization

In [188]:
doc2vec = Doc2VecSeqVect(params = doc2vec_params)

2025-07-04 15:57:26,439 : INFO : adding document #0 to Dictionary<0 unique tokens: []>
2025-07-04 15:57:26,460 : INFO : built Dictionary<1652 unique tokens: ['ag', 'ari', 'iv', 'pt', '▁activ']...> from 242 documents (total 78439 corpus positions)
2025-07-04 15:57:26,461 : INFO : Dictionary lifecycle event {'msg': "built Dictionary<1652 unique tokens: ['ag', 'ari', 'iv', 'pt', '▁activ']...> from 242 documents (total 78439 corpus positions)", 'datetime': '2025-07-04T15:57:26.461437', 'gensim': '4.3.3', 'python': '3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 18:02:02) \n[Clang 18.1.8 ]', 'platform': 'macOS-15.5-arm64-arm-64bit', 'event': 'created'}
2025-07-04 15:57:26,461 : INFO : conventional preprocessing documents and dictionary
2025-07-04 15:57:26,464 : INFO : loading Doc2Vec object from ../dvc-data/systems/dronology/2vec_models/bpe128k/doc2vec/[doc2vec-py-java-PVDBOW-500-20E-128k-c-1751384704.291919].model


FileNotFoundError: [Errno 2] No such file or directory: '../dvc-data/systems/dronology/2vec_models/bpe128k/doc2vec/[doc2vec-py-java-PVDBOW-500-20E-128k-c-1751384704.291919].model'

In [189]:
#[step1]Apply Doc2Vec Inference
doc2vec.InferDoc2Vec(steps=200)

NameError: name 'doc2vec' is not defined

In [190]:
doc2vec.df_inferred_src.head(2)

NameError: name 'doc2vec' is not defined

In [191]:
#test_inferDoc2Vec_trg = inferDoc2Vec(df_target)
#test_inferDoc2Vec_trg.head()
doc2vec.df_inferred_trg.head(2)

NameError: name 'doc2vec' is not defined

In [192]:
doc2vec.df_inferred_src .reset_index(drop=True, inplace=True)
doc2vec.df_inferred_trg .reset_index(drop=True, inplace=True)

(doc2vec.df_inferred_trg['inf-doc2vec'][0], doc2vec.df_inferred_trg['inf-doc2vec'][0])

NameError: name 'doc2vec' is not defined

In [193]:
#NOTE: Used .py file to compute, notebook was crashing
#[step 2]NonGroundTruth Computation
#metric_l = [DistanceMetric.EUC,DistanceMetric.COS,DistanceMetric.MAN]# , SimilarityMetric.Pearson]
#doc2vec.ComputeDistanceArtifacts( sampling=False, samples = 50, metric_list = metric_l )
#doc2vec.df_nonground_link.head()

In [ ]:
#[step 3]Saving Non-GroundTruth Links
#doc2vec.SaveLinks()

In [56]:
#export
def LoadLinks(timestamp, params, grtruth=False, sep=' ' ):
    
    path= params['saving_path'] + '['+ params['system'] + '-' + str(params['vectorizationType']) + '-' + str(params['linkType']) + '-' + str(grtruth) + '-{}].csv'.format(timestamp)
    
    logging.info("Loading computed links from... "+ path)

    return pd.read_csv(path, header=0, index_col=0, sep=sep)

In [58]:
big = pd.read_csv('../dvc-data/systems/dronology/experiment4.1.3/results/dronology_doc2vec_all_with_labels_20250704132648.csv')

# 2) Split into ground vs non-ground
df_ground   = big[ big['Linked?'] == 1 ].copy()
df_nonground= big[ big['Linked?'] == 0 ].copy()

# 3) Write out two CSVs
out_dir = '../dvc-data/systems/dronology/experiment4.1.3/results/'
base    = 'dronology-VectorizationType.doc2vec-LinkType.req2src'

ts = '20250704132648'  # same timestamp as your big CSV
fn1 = f'{base}-True-{ts}.csv'
fn2 = f'{base}-False-{ts}.csv'

df_ground.to_csv(os.path.join(out_dir, fn1), index=False)
df_nonground.to_csv(os.path.join(out_dir, fn2), index=False)

print('Wrote:')
print('  Ground-truth links:', fn1, '(', len(df_ground), 'rows )')
print('  Non-ground links :', fn2, '(', len(df_nonground), 'rows )')

Wrote:
  Ground-truth links: dronology-VectorizationType.doc2vec-LinkType.req2src-True-20250704132648.csv ( 393 rows )
  Non-ground links : dronology-VectorizationType.doc2vec-LinkType.req2src-False-20250704132648.csv ( 10279 rows )


In [98]:
#Loading Non-GroundTruth Links (change the timestamp with the assigned in the previous step)
df_nonglinks_doc2vec = LoadLinks(timestamp=20250623132856, params=doc2vec_params, sep = ',')
df_nonglinks_doc2vec.head()

2025-07-02 07:58:18,490 : INFO : Loading computed links from... ../dvc-data/systems/dronology/experiment4.1.1/results/[dronology-VectorizationType.doc2vec-LinkType.req2src-False-20250623132856].csv


FileNotFoundError: [Errno 2] No such file or directory: '../dvc-data/systems/dronology/experiment4.1.1/results/[dronology-VectorizationType.doc2vec-LinkType.req2src-False-20250623132856].csv'

In [99]:
#[step 4]GroundTruthMatching Testing
path_to_ground_truth = '../dvc-data/systems/dronolgy/mappings/ground_truth.txt'
doc2vec.MatchWithGroundTruth(path_to_ground_truth)
doc2vec.df_ground_link

NameError: name 'doc2vec' is not defined

In [100]:
#[step 5]Saving GroundTruth Links
doc2vec.SaveLinks(grtruth = True)

NameError: name 'doc2vec' is not defined

In [101]:
#Loading Non-GroundTruth Links (change the timestamp with the assigned in the previous step)
df_glinks_doc2vec = LoadLinks(timestamp=20250623132856, params=doc2vec_params, grtruth = True, sep = ',')
df_glinks_doc2vec.head()

2025-07-02 08:02:23,104 : INFO : Loading computed links from... ../dvc-data/systems/dronology/experiment4.1.1/results/[dronology-VectorizationType.doc2vec-LinkType.req2src-True-20250623132856].csv


,Target,EUC_dist,EUC_sim,COS_dist,COS_sim,MAN_dist,MAN_sim,Linked?
Source,,,,,,,,
/Users/loganfecko/Desktop/Summer_Research/traceXplainer/dvc-data/systems/dronology/corpus/req/RE-616.txt,/Users/loganfecko/Desktop/Summer_Research/trac...,72.137848,0.013673,0.802002,0.197998,1298.8182,0.000769,1
/Users/loganfecko/Desktop/Summer_Research/traceXplainer/dvc-data/systems/dronology/corpus/req/RE-616.txt,/Users/loganfecko/Desktop/Summer_Research/trac...,51.365795,0.019096,0.809980,0.190020,941.0077,0.001062,1
/Users/loganfecko/Desktop/Summer_Research/traceXplainer/dvc-data/systems/dronology/corpus/req/RE-616.txt,/Users/loganfecko/Desktop/Summer_Research/trac...,65.371864,0.015067,0.808522,0.191478,1174.2109,0.000851,1
/Users/loganfecko/Desktop/Summer_Research/traceXplainer/dvc-data/systems/dronology/corpus/req/RE-616.txt,/Users/loganfecko/Desktop/Summer_Research/trac...,80.934586,0.012205,0.770840,0.229160,1460.6865,0.000684,1
/Users/loganfecko/Desktop/Summer_Research/traceXplainer/dvc-data/systems/dronology/corpus/req/RE-616.txt,/Users/loganfecko/Desktop/Summer_Research/trac...,85.800568,0.011521,0.814711,0.185289,1503.1764,0.000665,1


# Approach Evaluation and Interpretation (doc2vec)

In [111]:
#supervisedEvalDoc2vec = SupervisedVectorEvaluation(doc2vec, similarity=SimilarityMetric.EUC_sim)
#supervisedEvalDoc2vec = SupervisedVectorEvaluation(doc2vec, similarity=SimilarityMetric.COS_sim)
supervisedEvalDoc2vec = SupervisedVectorEvaluation(doc2vec, similarity=SimilarityMetric.MAN_sim)

NameError: name 'SupervisedVectorEvaluation' is not defined

In [ ]:
supervisedEvalDoc2vec.y_test

In [ ]:
supervisedEvalDoc2vec.y_score

In [ ]:
supervisedEvalDoc2vec.Compute_precision_recall_gain()

In [ ]:
supervisedEvalDoc2vec.Compute_avg_precision()

In [ ]:
supervisedEvalDoc2vec.Compute_roc_curve()

## Compute distribution of similarities doc2vec

In [ ]:
#Basic Statistics
filter_doc2vec = doc2vec.df_ground_link
filter_doc2vec.describe()

In [ ]:
lag_plot(filter_doc2vec[[SimilarityMetric.EUC_sim]])

In [ ]:
lag_plot(filter_doc2vec[DistanceMetric.EUC])

In [ ]:
filter_doc2vec.hist(column=[SimilarityMetric.EUC_sim,DistanceMetric.EUC],color='k',bins=50,figsize=[10,5],alpha=0.5)

In [ ]:
#Separate distance from similarity analysis here
errors = filter_doc2vec[[SimilarityMetric.EUC_sim,DistanceMetric.EUC]].std()
print(errors)
filter_doc2vec[[SimilarityMetric.EUC_sim,DistanceMetric.EUC]].plot.kde()

In [ ]:
filter_doc2vec.hist(by='Linked?',column=SimilarityMetric.EUC_sim,figsize=[10, 5],bins=80)

In [ ]:
filter_doc2vec.hist(by='Linked?',column=DistanceMetric.EUC,figsize=[10, 5],bins=80)

In [ ]:
#separate the distance from the similarity plot
boxplot = filter_doc2vec.boxplot(by='Linked?',column=[SimilarityMetric.EUC_sim,DistanceMetric.EUC],figsize=[10, 5])

In [ ]:
boxplot = filter_doc2vec.boxplot(by='Linked?',column=[SimilarityMetric.EUC_sim],figsize=[10, 5])

## Combining Doc2vec and Word2vec
Please check this post for futher detatils [link](https://stats.stackexchange.com/questions/217614/intepreting-doc2vec-cosine-similarity-between-doc-vectors-and-word-vectors)

In [ ]:
! nbdev_build_docs #<-------- [Activate when stable]

In [ ]:
! nbdev_build_lib

In [ ]:
from nbdev.export import notebook2script
notebook2script()

In [ ]:
#! pip install -e .